In [1]:
import pandas as pd

credit_df = pd.read_csv("../data/raw/creditcard.csv")
X_fraud_train = pd.read_csv("../data/processed/resampled_training_data_X.csv")
y_fraud_train = pd.read_csv("../data/processed/resampled_training_data_y.csv")
X_fraud_test = pd.read_csv("../data/processed/test_data_X.csv")
y_fraud_test = pd.read_csv("../data/processed/test_data_y.csv")


In [2]:
from sklearn.model_selection import train_test_split

X_credit = credit_df.drop(columns=['Class'])
y_credit = credit_df['Class']
X_credit_train, X_credit_test, y_credit_train, y_credit_test = train_test_split(
    X_credit, y_credit, test_size=0.2, random_state=42, stratify=y_credit
)

In [3]:
import joblib
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, average_precision_score, precision_recall_curve

def train_and_evaluate(X_train, X_test, y_train, y_test, dataset_name):
    print(f"\n====== {dataset_name.upper()} ======")

    # Logistic Regression
    lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr.fit(X_train, y_train)
    y_pred_lr = lr.predict(X_test)
    y_prob_lr = lr.predict_proba(X_test)[:, 1]

    # XGBoost
    xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    xgb.fit(X_train, y_train)
    y_pred_xgb = xgb.predict(X_test)
    y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

    joblib.dump(xgb, "../outputs/xgb_fraud_model.pkl")
    joblib.dump(lr, "../outputs/lr_fraud_model.pkl")



    # Evaluation
    for name, y_pred, y_prob in zip(['Logistic Regression', 'XGBoost'],
                                    [y_pred_lr, y_pred_xgb],
                                    [y_prob_lr, y_prob_xgb]):
        print(f"\n📊 Model: {name}")
        print("Confusion Matrix:")
        print(confusion_matrix(y_test, y_pred))
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))
        print("F1 Score:", f1_score(y_test, y_pred))
        print("AUC-PR:", average_precision_score(y_test, y_prob))


In [4]:
train_and_evaluate(X_credit_train, X_credit_test, y_credit_train, y_credit_test, "Credit Card")
train_and_evaluate(X_fraud_train, X_fraud_test, y_fraud_train, y_fraud_test, "Fraud Data")


====== CREDIT CARD ======


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:183: UserWarning: [19:40:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



📊 Model: Logistic Regression
Confusion Matrix:
[[54665  2199]
 [    8    90]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.96      0.98     56864
           1       0.04      0.92      0.08        98

    accuracy                           0.96     56962
   macro avg       0.52      0.94      0.53     56962
weighted avg       1.00      0.96      0.98     56962

F1 Score: 0.07540846250523671
AUC-PR: 0.6708643561829816

📊 Model: XGBoost
Confusion Matrix:
[[56852    12]
 [   20    78]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.87      0.80      0.83        98

    accuracy                           1.00     56962
   macro avg       0.93      0.90      0.91     56962
weighted avg       1.00      1.00      1.00     56962

F1 Score: 0.8297872340425533
AUC-PR: 0.7972912780329892

====== FRAUD DATA ======


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:183: UserWarning: [19:41:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



📊 Model: Logistic Regression
Confusion Matrix:
[[17838  9555]
 [  843  1987]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.65      0.77     27393
           1       0.17      0.70      0.28      2830

    accuracy                           0.66     30223
   macro avg       0.56      0.68      0.53     30223
weighted avg       0.88      0.66      0.73     30223

F1 Score: 0.27650988032285
AUC-PR: 0.5661969066195596

📊 Model: XGBoost
Confusion Matrix:
[[25879  1514]
 [ 1223  1607]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.94      0.95     27393
           1       0.51      0.57      0.54      2830

    accuracy                           0.91     30223
   macro avg       0.73      0.76      0.74     30223
weighted avg       0.91      0.91      0.91     30223

F1 Score: 0.5400772979331204
AUC-PR: 0.6206655366525421
